# Mortgage Mathematics and Mortgage-Backed Securities


## 1. Big picture: why mortgage-backed securities matter

Mortgage-backed securities (MBS) are a type of **asset-backed security (ABS)**.  Instead of being backed by one borrower, an MBS is backed by a pool of mortgage loans.  The PDF notes that the mortgage market is very large and historically important in fixed-income markets.  The main idea is **securitization**:

1. Many individual mortgages are pooled together.
2. The cash flows from that pool are redirected to investors.
3. The MBS may be split into different tranches with different risk and timing profiles.

The simplest security is a **pass-through MBS**, where mortgage cash flows are passed through to investors.  More complex structures include **principal-only (PO)**, **interest-only (IO)**, and **collateralized mortgage obligations (CMOs)**.

The foundation for all of these products is the cash-flow mathematics of the underlying level-payment mortgage.

## 2. Level-payment mortgage mathematics

A **level-payment mortgage** has:

- Initial principal: $M_0$.
- Fixed periodic payment: $B$.
- Periodic mortgage coupon rate: $c$.
- Total number of repayment periods: $n$.

After each payment, the outstanding balance earns interest and then the borrower pays $B$.  The balance recursion is:

$$
M_k = (1+c)M_{k-1} - B, \qquad k=1,2,\ldots,n.
$$

The mortgage is **fully amortizing** when the final balance is zero:

$$
M_n = 0.
$$

### 2.1 Deriving the mortgage payment

Iterating the recursion gives:

$$
M_k = (1+c)^kM_0 - B\sum_{p=0}^{k-1}(1+c)^p.
$$

The geometric sum is:

$$
\sum_{p=0}^{k-1}(1+c)^p = \frac{(1+c)^k - 1}{c}.
$$

So:

$$
M_k = (1+c)^kM_0 - B\frac{(1+c)^k - 1}{c}.
$$

Set $k=n$ and use $M_n=0$:

$$
0 = (1+c)^nM_0 - B\frac{(1+c)^n - 1}{c}.
$$

Solving for $B$ gives the standard fixed-payment formula:

$$
B = \frac{c(1+c)^n}{(1+c)^n - 1}M_0.
$$

### 2.2 Outstanding balance formula

Substituting the payment formula back into the balance equation gives:

$$
M_k = M_0\frac{(1+c)^n - (1+c)^k}{(1+c)^n - 1}.
$$

### 2.3 Interest and scheduled principal

In period $k$, the interest due is:

$$
I_k = cM_{k-1}.
$$

The scheduled principal repayment is the part of the payment that reduces principal:

$$
P_k = B - I_k = B - cM_{k-1}.
$$

Early in the mortgage, $M_{k-1}$ is large, so most of $B$ goes to interest.  Later, the balance is lower, so more of $B$ goes to principal.

In [1]:
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.6f}')


def level_payment(principal: float, periodic_rate: float, n_periods: int) -> float:
    """Fixed payment for a fully amortizing level-payment mortgage."""
    if n_periods <= 0:
        raise ValueError('n_periods must be positive')
    if abs(periodic_rate) < 1e-15:
        return principal / n_periods
    return principal * periodic_rate * (1 + periodic_rate) ** n_periods / ((1 + periodic_rate) ** n_periods - 1)


def mortgage_balance_closed_form(principal: float, periodic_rate: float, n_periods: int, k: int) -> float:
    """Outstanding mortgage balance after k payments using the closed-form formula."""
    if k < 0 or k > n_periods:
        raise ValueError('k must be between 0 and n_periods')
    if abs(periodic_rate) < 1e-15:
        return principal * (1 - k / n_periods)
    numerator = (1 + periodic_rate) ** n_periods - (1 + periodic_rate) ** k
    denominator = (1 + periodic_rate) ** n_periods - 1
    return principal * numerator / denominator


def amortization_schedule(principal: float, annual_rate: float, term_months: int) -> pd.DataFrame:
    """Monthly amortization schedule for a level-payment mortgage."""
    c = annual_rate / 12
    B = level_payment(principal, c, term_months)
    balance = principal
    rows = []
    for month in range(1, term_months + 1):
        interest = c * balance
        scheduled_principal = B - interest
        ending_balance = max(0.0, balance - scheduled_principal)
        rows.append({
            'month': month,
            'beginning_balance': balance,
            'payment': B,
            'interest': interest,
            'scheduled_principal': scheduled_principal,
            'ending_balance': ending_balance,
        })
        balance = ending_balance
    return pd.DataFrame(rows)

## 3. Single-mortgage cash flows from the Excel workbook

The `SingleMortgageCashFlows` worksheet uses the following assumptions:

| Input | Value |
|---|---:|
| Mortgage loan | $100,000 |
| Annual mortgage rate | 8.125% |
| Term | 360 months |

The workbook's monthly payment is approximately **$742.50**.  The code below reproduces the schedule.

In [2]:
single_schedule = amortization_schedule(principal=100_000, annual_rate=0.08125, term_months=360)
print('Monthly payment:', round(single_schedule.loc[0, 'payment'], 2))
single_schedule.head(12)

Monthly payment: 742.5


,month,beginning_balance,payment,interest,scheduled_principal,ending_balance
0,1,"100,000.000000",742.497180,677.083333,65.413847,"99,934.586153"
1,2,"99,934.586153",742.497180,676.640427,65.856753,"99,868.729399"
2,3,"99,868.729399",742.497180,676.194522,66.302659,"99,802.426741"
3,4,"99,802.426741",742.497180,675.745598,66.751583,"99,735.675158"
4,5,"99,735.675158",742.497180,675.293634,67.203547,"99,668.471612"
5,6,"99,668.471612",742.497180,674.838610,67.658571,"99,600.813041"
6,7,"99,600.813041",742.497180,674.380505,68.116676,"99,532.696365"
7,8,"99,532.696365",742.497180,673.919298,68.577882,"99,464.118483"
8,9,"99,464.118483",742.497180,673.454969,69.042212,"99,395.076272"
9,10,"99,395.076272",742.497180,672.987496,69.509685,"99,325.566587"


## 4. Present value of a level-payment mortgage

If there are no defaults and no prepayments, and if the periodic risk-free discount rate is $r$, the present value of the mortgage payment stream is:

$$
F_0 = \sum_{k=1}^{n}\frac{B}{(1+r)^k}.
$$

Using the fixed-payment formula:

$$
F_0
= \frac{c(1+c)^nM_0}{(1+c)^n - 1}\cdot\frac{(1+r)^n - 1}{r(1+r)^n}.
$$

A useful check is that if the discount rate equals the mortgage coupon rate, $r=c$, then the mortgage value equals the principal:

$$
F_0 = M_0.
$$

In practice, the mortgage coupon is often above the risk-free rate because it includes compensation for servicing fees, prepayment uncertainty, default risk, and profit margins.

In [3]:
def pv_level_mortgage(principal: float, mortgage_periodic_rate: float, discount_periodic_rate: float, n_periods: int) -> float:
    """Present value of a level-payment mortgage assuming no default and no prepayment."""
    B = level_payment(principal, mortgage_periodic_rate, n_periods)
    months = np.arange(1, n_periods + 1)
    discount_factors = 1 / (1 + discount_periodic_rate) ** months
    return float(np.sum(B * discount_factors))

# Check: if r = c, PV should equal principal.
pv_check = pv_level_mortgage(100_000, 0.08125 / 12, 0.08125 / 12, 360)
pv_check

99999.99999999913

## 5. Prepayment risk and PSA convention

Mortgage borrowers can pay extra principal earlier than scheduled.  These extra payments are called **prepayments**.  They matter because prepayment changes the timing and amount of cash flows received by MBS investors.

Important prepayment measures:

### Conditional Prepayment Rate (CPR)

CPR is the annualized rate at which a mortgage pool prepays, expressed as a percentage of the current outstanding balance.

### Single-Month Mortality Rate (SMM)

SMM is the monthly version of CPR:

$$
\text{SMM} = 1 - (1 - \text{CPR})^{1/12}.
$$

The reverse conversion is:

$$
\text{CPR} = 1 - (1 - \text{SMM})^{12}.
$$

### PSA benchmark

The 100 PSA benchmark assumes that CPR ramps up linearly to 6% over the first 30 months and then remains at 6%:

$$
\text{CPR}_t =
\begin{cases}
0.06\times\frac{t}{30}, & t \le 30,\\
0.06, & t > 30.
\end{cases}
$$

A 200 PSA assumption doubles the CPR path.  A 50 PSA assumption halves it.

In [4]:
def cpr_to_smm(cpr: float) -> float:
    """Convert annual CPR to monthly SMM."""
    if cpr < 0:
        raise ValueError('CPR cannot be negative')
    if cpr >= 1:
        return 1.0
    return 1 - (1 - cpr) ** (1 / 12)


def smm_to_cpr(smm: float) -> float:
    """Convert monthly SMM to annual CPR."""
    if smm < 0:
        raise ValueError('SMM cannot be negative')
    if smm >= 1:
        return 1.0
    return 1 - (1 - smm) ** 12


def psa_cpr(month: int, seasoning_months: int = 0, psa_multiple: float = 1.0) -> float:
    """
    CPR under the PSA benchmark.
    month is the cash-flow month starting at 1.
    seasoning_months is the age of the pool at month 0.
    """
    age = month + seasoning_months
    base_cpr = 0.06 * min(age / 30, 1.0)
    return psa_multiple * base_cpr

psa_demo = pd.DataFrame({
    'month': range(1, 13),
    'CPR_100_PSA': [psa_cpr(m, psa_multiple=1.0) for m in range(1, 13)],
    'SMM_100_PSA': [cpr_to_smm(psa_cpr(m, psa_multiple=1.0)) for m in range(1, 13)],
})
psa_demo

,month,CPR_100_PSA,SMM_100_PSA
0,1,0.002000,0.000167
1,2,0.004000,0.000334
2,3,0.006000,0.000501
3,4,0.008000,0.000669
4,5,0.010000,0.000837
5,6,0.012000,0.001006
6,7,0.014000,0.001174
7,8,0.016000,0.001343
8,9,0.018000,0.001513
9,10,0.020000,0.001682


## 6. Mortgage pass-through MBS

A **pass-through MBS** pools mortgages and passes through monthly cash flows to investors.  Investors receive interest plus principal payments.  Principal payments include both scheduled principal and prepayments.

The pass-through coupon rate is usually lower than the weighted-average coupon rate of the underlying mortgage pool because servicing and guarantee fees are deducted.

### Key pool-level definitions

**Weighted Average Coupon (WAC)** is the weighted-average mortgage coupon of the loans in the pool:

$$
\text{WAC} = \sum_i w_i c_i,
$$

where $w_i$ is based on each loan's outstanding balance.

**Weighted Average Maturity (WAM)** is the weighted-average remaining term to maturity:

$$
\text{WAM} = \sum_i w_i n_i.
$$

### Excel-style monthly cash-flow logic

The `Pass-Through` worksheet uses this logic each month:

1. Beginning balance is the previous month's ending balance.
2. Monthly payment is recalculated from the beginning balance and remaining term.
3. Mortgage interest paid by borrowers is mortgage coupon / 12 times beginning balance.
4. Investor interest is pass-through coupon / 12 times beginning balance.
5. Scheduled principal is payment minus mortgage interest.
6. CPR is based on PSA and seasoning.
7. Prepayment is:

$$
\text{Prepayment}_k = (M_{k-1} - \text{ScheduledPrincipal}_k)\times \text{SMM}_k.
$$

8. Total principal payment is scheduled principal plus prepayment.
9. Ending balance is beginning balance minus total principal.

In [5]:
def pass_through_cashflows(
    principal: float,
    mortgage_annual_rate: float,
    pass_through_annual_rate: float,
    term_months: int,
    seasoning_months: int = 0,
    psa_multiple: float = 1.0,
) -> pd.DataFrame:
    """Replicate the Excel-style pass-through cash-flow model."""
    balance = float(principal)
    rows = []
    for month in range(1, term_months - seasoning_months + 1):
        if balance < 1e-8:
            break
        remaining_term = term_months - seasoning_months - month + 1
        monthly_payment = level_payment(balance, mortgage_annual_rate / 12, remaining_term)
        mortgage_interest = balance * mortgage_annual_rate / 12
        investor_interest = balance * pass_through_annual_rate / 12
        scheduled_principal = monthly_payment - mortgage_interest
        cpr = psa_cpr(month, seasoning_months=seasoning_months, psa_multiple=psa_multiple)
        smm = cpr_to_smm(cpr)
        prepayment = (balance - scheduled_principal) * smm
        # prevent tiny negative due to numerical rounding in final month
        prepayment = max(prepayment, 0.0)
        total_principal = min(balance, scheduled_principal + prepayment)
        ending_balance = max(0.0, balance - total_principal)
        rows.append({
            'month': month,
            'CPR': cpr,
            'SMM': smm,
            'beginning_balance': balance,
            'monthly_payment': monthly_payment,
            'mortgage_interest_paid_by_borrowers': mortgage_interest,
            'investor_interest': investor_interest,
            'scheduled_principal': scheduled_principal,
            'prepayment': total_principal - scheduled_principal,
            'total_principal': total_principal,
            'ending_balance': ending_balance,
        })
        balance = ending_balance
    df = pd.DataFrame(rows)
    df.attrs['total_investor_interest'] = df['investor_interest'].sum()
    df.attrs['total_prepayments'] = df['prepayment'].sum()
    df.attrs['total_principal'] = df['total_principal'].sum()
    df.attrs['average_life_years'] = (df['month'] * df['total_principal']).sum() / (12 * df['total_principal'].sum())
    return df

# Replicate the Excel Pass-Through sheet assumptions.
pt_excel = pass_through_cashflows(
    principal=400,
    mortgage_annual_rate=0.08125,
    pass_through_annual_rate=0.075,
    term_months=360,
    seasoning_months=3,
    psa_multiple=1.0,
)

print('Average life (years):', round(pt_excel.attrs['average_life_years'], 6))
print('Total investor interest:', round(pt_excel.attrs['total_investor_interest'], 6))
print('Total prepayments:', round(pt_excel.attrs['total_prepayments'], 6))
pt_excel.head(12)

Average life (years): 11.67101
Total investor interest: 350.130301
Total prepayments: 263.559219


,month,CPR,SMM,beginning_balance,monthly_payment,mortgage_interest_paid_by_borrowers,investor_interest,scheduled_principal,prepayment,total_principal,ending_balance
0,1,0.008000,0.000669,400.000000,2.975868,2.708333,2.500000,0.267535,0.267470,0.535005,399.464995
1,2,0.010000,0.000837,399.464995,2.973877,2.704711,2.496656,0.269166,0.334198,0.603364,398.861631
2,3,0.012000,0.001006,398.861631,2.971387,2.700626,2.492885,0.270762,0.400800,0.671562,398.190069
3,4,0.014000,0.001174,398.190069,2.968399,2.696079,2.488688,0.272321,0.467243,0.739564,397.450505
4,5,0.016000,0.001343,397.450505,2.964914,2.691071,2.484066,0.273843,0.533493,0.807335,396.643170
5,6,0.018000,0.001513,396.643170,2.960931,2.685605,2.479020,0.275327,0.599514,0.874841,395.768329
6,7,0.020000,0.001682,395.768329,2.956453,2.679681,2.473552,0.276772,0.665273,0.942045,394.826284
7,8,0.022000,0.001852,394.826284,2.951480,2.673303,2.467664,0.278177,0.730736,1.008913,393.817371
8,9,0.024000,0.002022,393.817371,2.946013,2.666472,2.461359,0.279542,0.795869,1.075410,392.741961
9,10,0.026000,0.002193,392.741961,2.940056,2.659190,2.454637,0.280865,0.860637,1.141502,391.600459


## 7. Average life, yields, contraction risk, and extension risk

The **average life** of an MBS is the weighted-average time to receive principal payments:

$$
\text{Average Life} = \frac{\sum_{k=1}^{T} kP_k}{12\times TP},
$$

where:

- $P_k$ is the principal paid in month $k$.
- $TP$ is total principal.
- The division by 12 converts months into years.

When PSA speed increases, principal is returned earlier, so average life decreases.

### Mortgage yields

MBS yields depend on an assumed prepayment model.  A quoted yield under 100 PSA may differ from a yield under 300 PSA because the expected cash-flow timing changes.  This is why the slides emphasize that a mortgage yield is meaningful only together with its prepayment assumption.

### Contraction risk

When interest rates fall, borrowers are more likely to refinance.  Prepayments increase, principal comes back early, and investors must reinvest at lower yields.  This is **contraction risk**.

### Extension risk

When interest rates rise, borrowers are less likely to refinance.  Prepayments slow, principal comes back later, and investors remain locked into the older mortgage cash flows for longer.  This is **extension risk**.

In [6]:
# Compare average life under different PSA speeds for the Excel pass-through example.
psa_comparison = []
for psa in [0.5, 1.0, 2.0, 3.0]:
    df = pass_through_cashflows(
        principal=400,
        mortgage_annual_rate=0.08125,
        pass_through_annual_rate=0.075,
        term_months=360,
        seasoning_months=3,
        psa_multiple=psa,
    )
    psa_comparison.append({
        'PSA multiple': psa,
        'average_life_years': df.attrs['average_life_years'],
        'total_investor_interest': df.attrs['total_investor_interest'],
        'total_prepayments': df.attrs['total_prepayments'],
    })

pd.DataFrame(psa_comparison)

,PSA multiple,average_life_years,total_investor_interest,total_prepayments
0,0.500000,15.128434,453.853017,171.735514
1,1.000000,11.671010,350.130301,263.559219
2,2.000000,7.694256,230.827687,341.681397
3,3.000000,5.643368,169.301037,367.981063


## 8. Principal-only and interest-only MBS

Because each mortgage payment can be split into principal and interest, an MBS can also be split into:

- **Principal-only (PO):** receives principal cash flows.
- **Interest-only (IO):** receives interest cash flows.

From the mortgage formulas:

$$
I_k = cM_{k-1},
$$

and:

$$
P_k = B - cM_{k-1}.
$$

Using the closed-form mortgage balance, the scheduled principal can be written as:

$$
P_k = (B - cM_0)(1+c)^{k-1}.
$$

### Present value of a PO stream

Assuming no prepayments and discount rate $r$, the PO present value is:

$$
V_0 = \sum_{k=1}^{n}\frac{P_k}{(1+r)^k}.
$$

The slide formula can be written as:

$$
V_0 = (B - cM_0)\frac{(1+r)^n - (1+c)^n}{(r-c)(1+r)^n}.
$$

### Present value of an IO stream

The IO present value is the present value of the full mortgage minus the PO value:

$$
W_0 = F_0 - V_0.
$$

### Intuition

- PO investors like faster prepayment because they receive principal sooner.
- IO investors dislike faster prepayment because prepayment reduces the remaining balance and therefore reduces future interest.

In [7]:
def po_io_cashflows(principal: float, annual_rate: float, term_months: int, discount_annual_rate: float) -> dict:
    """PO and IO values and durations assuming no prepayments."""
    sched = amortization_schedule(principal, annual_rate, term_months)
    r = discount_annual_rate / 12
    months = sched['month'].to_numpy()
    discount = 1 / (1 + r) ** months
    principal_cf = sched['scheduled_principal'].to_numpy()
    interest_cf = sched['interest'].to_numpy()
    po_value = float(np.sum(principal_cf * discount))
    io_value = float(np.sum(interest_cf * discount))
    total_value = po_value + io_value
    po_duration = float(np.sum(months * principal_cf * discount) / (12 * po_value))
    io_duration = float(np.sum(months * interest_cf * discount) / (12 * io_value))
    return {
        'PO_value': po_value,
        'IO_value': io_value,
        'total_mortgage_value': total_value,
        'PO_duration_years': po_duration,
        'IO_duration_years': io_duration,
    }

po_io_cashflows(100_000, annual_rate=0.08125, term_months=360, discount_annual_rate=0.08125)

{'PO_value': 23390.611045706537,
 'IO_value': 76609.3889542926,
 'total_mortgage_value': 99999.99999999914,
 'PO_duration_years': 15.0416666666666,
 'IO_duration_years': 7.798458342399955}

## 9. Duration of PO and IO cash flows

Duration is the weighted-average time at which cash flows are received.  It is a standard measure of timing risk.

For the PO stream:

$$
D_P = \frac{1}{12V_0}\sum_{k=1}^{n}\frac{kP_k}{(1+r)^k}.
$$

For the IO stream:

$$
D_I = \frac{1}{12W_0}\sum_{k=1}^{n}\frac{kI_k}{(1+r)^k}.
$$

The PO stream usually has longer duration than the IO stream because scheduled principal is received more heavily later in the mortgage life.  The IO stream is front-loaded because interest is highest when the outstanding balance is high.

## 10. Sequential-pay CMO

A **collateralized mortgage obligation (CMO)** redirects cash flows from mortgage securities into different tranches.  The slides highlight that CMOs were created partly to reshape prepayment risk and produce securities better suited to different investors.

The simplest structure is a **sequential-pay CMO**:

1. Each tranche receives coupon interest based on its beginning-of-period outstanding balance.
2. All principal first goes to tranche A until A is retired.
3. Then principal goes to tranche B until B is retired.
4. Then C, then D, and so on.

The Excel workbook uses these tranches:

| Tranche | Par amount | Coupon |
|---|---:|---:|
| A | 194.5 | 7.5% |
| B | 36.0 | 7.5% |
| C | 96.5 | 7.5% |
| D | 73.0 | 7.5% |
| Total | 400.0 | 7.5% |

In [8]:
def sequential_pay_cmo(pool_cashflows: pd.DataFrame, tranche_pars: dict, tranche_annual_coupons: dict) -> dict:
    """Allocate pool principal sequentially across CMO tranches."""
    balances = {name: float(par) for name, par in tranche_pars.items()}
    rows = []
    for _, pool_row in pool_cashflows.iterrows():
        month = int(pool_row['month'])
        principal_available = float(pool_row['total_principal'])
        row = {'month': month}
        for name in tranche_pars:
            beg_bal = balances[name]
            interest = beg_bal * tranche_annual_coupons[name] / 12
            principal_paid = min(beg_bal, principal_available)
            balances[name] = max(0.0, beg_bal - principal_paid)
            principal_available -= principal_paid
            row[f'{name}_beginning_balance'] = beg_bal
            row[f'{name}_principal'] = principal_paid
            row[f'{name}_interest'] = interest
            row[f'{name}_ending_balance'] = balances[name]
        rows.append(row)
    cmo_df = pd.DataFrame(rows)
    wal = {}
    for name, par in tranche_pars.items():
        principal_col = f'{name}_principal'
        total_principal = cmo_df[principal_col].sum()
        wal[name] = (cmo_df['month'] * cmo_df[principal_col]).sum() / (12 * total_principal)
    return {'cashflows': cmo_df, 'weighted_average_life': wal}

tranche_pars = {'A': 194.5, 'B': 36.0, 'C': 96.5, 'D': 73.0}
tranche_coupons = {'A': 0.075, 'B': 0.075, 'C': 0.075, 'D': 0.075}

cmo_result = sequential_pay_cmo(pt_excel, tranche_pars, tranche_coupons)
cmo_wal = pd.DataFrame({
    'tranche': list(cmo_result['weighted_average_life'].keys()),
    'WAL_years': list(cmo_result['weighted_average_life'].values()),
})
cmo_wal

,tranche,WAL_years
0,A,4.918270
1,B,10.876730
2,C,15.798284
3,D,24.598684


The weighted-average lives should be close to the workbook values:

| Tranche | Workbook WAL (years) | Interpretation |
|---|---:|---|
| A | 4.918 | First tranche to receive principal, shortest life. |
| B | 10.877 | Receives principal after A is retired. |
| C | 15.798 | Receives principal after A and B are retired. |
| D | 24.599 | Last principal tranche, longest life. |

This shows how CMO structuring can transform one mortgage pool into securities with very different timing risks.

## 11. Pricing MBS and prepayment modeling

The slides emphasize that **prepayment modeling is one of the most important parts of any residential MBS pricing engine**.  Term-structure models are relatively standard, but prepayment models are difficult because borrower behavior is hard to observe and calibrate.

One model mentioned in the slides is the Richard and Roll (1989) style multiplicative CPR model:

$$
\text{CPR}_k = RI_k \times AGE_k \times MM_k \times BM_k.
$$

Where:

- $RI_k$ is the refinancing incentive.
- $AGE_k$ is a seasoning multiplier.
- $MM_k$ is a monthly multiplier.
- $BM_k$ is a burnout multiplier.

A refinancing incentive example is:

$$
RI_k = 0.28 + 0.14\tan^{-1}\left(-8.57 + 430(WAC - r_k(10))\right),
$$

where $r_k(10)$ is the prevailing 10-year spot rate.

A seasoning multiplier example is:

$$
AGE_k = \min\left(1, \frac{t}{30}\right).
$$

A burnout multiplier example is:

$$
BM_k = 0.3 + 0.7\frac{M_{k-1}}{M_0}.
$$

The term-structure model is needed to discount cash flows and to compute interest-rate inputs such as the 10-year spot rate.  Because prepayment behavior changes with interest rates, full MBS pricing often requires Monte Carlo simulation.

In [9]:
def richard_roll_cpr(wac: float, r10: float, age_month: int, month_multiplier: float, balance_prev: float, original_balance: float) -> float:
    """Simple implementation of the Richard-Roll style CPR components from the slides."""
    refinancing_incentive = 0.28 + 0.14 * np.arctan(-8.57 + 430 * (wac - r10))
    age_multiplier = min(1.0, age_month / 30)
    burnout_multiplier = 0.3 + 0.7 * (balance_prev / original_balance)
    cpr = refinancing_incentive * age_multiplier * month_multiplier * burnout_multiplier
    return float(np.clip(cpr, 0.0, 1.0))

# Example: WAC is above the 10-year spot rate, creating refinance incentive.
richard_roll_cpr(
    wac=0.08,
    r10=0.05,
    age_month=18,
    month_multiplier=1.0,
    balance_prev=380,
    original_balance=400,
)

0.27105076384411325

## 12. Practice / test question answers with Python



### 12.1 level-payment mortgage and pass-through MBS

The question instructions say to divide annual rates by 12 and assume monthly compounding.

#### Question 1

Compute the monthly payment for a 30-year mortgage with:

- Principal: $400,000$
- Annual mortgage rate: 5%
- Term: 360 months

Formula:

$$
B = \frac{c(1+c)^n}{(1+c)^n - 1}M_0,
$$

where $c=0.05/12$ and $n=360$.

#### Questions 2-4

For the pass-through questions:

- Pool balance: 400 million
- Mortgage coupon: 6% annual
- Pass-through coupon: 5% annual
- Term: 20 years = 240 months
- Seasoning: 0
- PSA: 100 PSA for Q2 and Q3; 200 PSA for Q4

This notebook uses the same Excel-style pass-through logic as the workbook: scheduled payment is recalculated from the current balance and remaining term each month.

In [ ]:
# Q1
q1_payment = level_payment(400_000, 0.05 / 12, 360)

#  Q2 and Q3: 100 PSA
q23 = pass_through_cashflows(
    principal=400,
    mortgage_annual_rate=0.06,
    pass_through_annual_rate=0.05,
    term_months=240,
    seasoning_months=0,
    psa_multiple=1.0,
)

# Q4: 200 PSA
# The question asks for TOTAL PREPAYMENTS, not investor interest.
q4 = pass_through_cashflows(
    principal=400,
    mortgage_annual_rate=0.06,
    pass_through_annual_rate=0.05,
    term_months=240,
    seasoning_months=0,
    psa_multiple=2.0,
)

screenshot_answers = pd.DataFrame([
    {'question': 'Q1 monthly payment', 'answer': q1_payment, 'unit': 'dollars'},
    {'question': 'Q2 total investor interest at 100 PSA', 'answer': q23.attrs['total_investor_interest'], 'unit': 'million dollars'},
    {'question': 'Q3 total prepayments at 100 PSA', 'answer': q23.attrs['total_prepayments'], 'unit': 'million dollars'},
    {'question': 'Q4 total prepayments at 200 PSA', 'answer': q4.attrs['total_prepayments'], 'unit': 'million dollars'},
])

screenshot_answers['rounded_answer'] = screenshot_answers['answer'].round(2)
screenshot_answers


,question,answer,unit,rounded_answer
0,Q1 monthly payment,"2,147.286492",dollars,"2,147.290000"
1,Q2 total investor interest at 100 PSA,171.176268,million dollars,171.180000
2,Q3 total prepayments at 100 PSA,181.090923,million dollars,181.090000
3,Q4 total prepayments at 200 PSA,268.150131,million dollars,268.150000


**Rounded answers for the screenshot questions**

| Question | Answer to submit |
|---|---:|
| Q1 monthly payment | 2147.29 |
| Q2 total investor interest at 100 PSA | 171.18 |
| Q3 total prepayments at 100 PSA | 181.09 |
| Q4 total prepayments at 200 PSA | 268.15 |

Important correction: Q4 asks for **total prepayments** when the prepayment speed increases to 200 PSA.  It does **not** ask for investor interest.  Under the Excel-style re-amortizing pass-through convention used in this notebook, the correct Q4 answer is **268.15** million.

Note: If a course platform uses a non-re-amortizing scheduled-payment convention instead of the Excel workbook convention, Q2-Q4 will differ.  The values above follow the workbook logic you uploaded.


### 13.2 PDF quiz examples

The PDF also contains several quiz screenshots.  The answers can be verified using Python.

#### Quiz set A: level-payment mortgage

1. In a deterministic world with no default or prepayment, if the risk-free discount rate equals the mortgage coupon rate ($r=c$), then the mortgage PV equals principal.  In practice, mortgage coupon rates are usually above risk-free rates because they include spreads and fees.
2. For $M_0=1{,}000{,}000$, $c=5\%$ per period and $n=20$, the level payment is about $80{,}242.59$.
3. After 15 payments, the remaining balance is about $347{,}408.40$; the principal in the 16th payment is about $62{,}872.17$.

In [12]:
# PDF quiz set A calculations
M0 = 1_000_000
c = 0.05
n = 20
B = level_payment(M0, c, n)
M15 = mortgage_balance_closed_form(M0, c, n, 15)
interest_16 = c * M15
principal_16 = B - interest_16

pd.DataFrame([
    {'item': 'level payment B', 'value': B},
    {'item': 'balance after 15 payments M15', 'value': M15},
    {'item': 'interest in payment 16', 'value': interest_16},
    {'item': 'principal in payment 16', 'value': principal_16},
]).assign(rounded=lambda d: d['value'].round(2))

,item,value,rounded
0,level payment B,"80,242.587191","80,242.590000"
1,balance after 15 payments M15,"347,408.409233","347,408.410000"
2,interest in payment 16,"17,370.420462","17,370.420000"
3,principal in payment 16,"62,872.166729","62,872.170000"


#### Quiz set B: average life and prepayment risk

1. If PSA speed increases, average life decreases because principal is returned earlier.
2. When interest rates decrease, prepayments tend to increase, creating contraction risk.
3. If 10%, 20%, 50%, and 20% of principal are paid at months 6, 9, 24, and 30, the average life is:

$$
\frac{6(0.10) + 9(0.20) + 24(0.50) + 30(0.20)}{12} = 1.70\text{ years}.
$$

In [13]:
avg_life_quiz = (6*0.10 + 9*0.20 + 24*0.50 + 30*0.20) / 12
avg_life_quiz

1.7

#### Quiz set C: PO/IO intuition

1. In early mortgage payments, most of the level payment goes to interest.
2. A larger duration means more timing risk.
3. An interest-only investor prefers prepayments to occur later, because the IO investor earns interest only while principal remains outstanding.

#### Quiz set D: sequential CMO and prepayment model

1. A tranche B investor can receive coupon interest before tranche A is retired, but cannot receive principal until tranche A is fully paid off.
2. In a sequential CMO, tranche A has shorter duration than tranche B: $d_A < d_B$.
3. If refinancing incentive increases, CPR tends to increase.  If the 10-year spot rate increases relative to WAC, refinancing incentive and CPR tend to decrease.  If CPR increases, mortgage duration usually decreases because principal is returned earlier.

## 14. Summary

This notebook covered the full workflow:

1. Start from level-payment mortgage mathematics.
2. Build single-loan amortization schedules.
3. Add prepayments using CPR, SMM and PSA.
4. Simulate mortgage pass-through cash flows.
5. Measure average life and prepayment risk.
6. Split mortgage cash flows into PO and IO securities.
7. Compute PO/IO value and duration.
8. Allocate pass-through principal into sequential-pay CMO tranches.
9. Discuss prepayment modeling and MBS pricing.
10. Solve the practice/test questions with Python.

The key conceptual point is that MBS valuation is not just about discounting fixed coupons.  Mortgage cash flows depend strongly on borrower prepayment behavior, which itself depends on interest rates, seasoning, refinancing incentives, seasonality and burnout.